# CATI Phase 2 — End-to-End Fine-tuning

| Step | Cell | Notes |
|------|------|-------|
| 0. Setup | 1 | Mount Drive, clone repo, install deps |
| 1. Deduplicate dataset | 2 | Remove duplicate frames by SHA-256 hash |
| 2. Prerequisites | 3 | Check Phase 1 ckpt + YOLO dataset |
| 3. Layer verification | 4 | Confirm neck FiLM layer indices |
| 4. Phase 2 training | 5 | Backbone + neck FiLM + aux loss |
| 5. Fair CATI eval | 6 | Evaluate with FiLM hooks active (correct method) |
| 6. Ablation | 7 | YOLO fine-tune without CATI |
| 7. Confidence sweep | 8 | Find optimal confidence threshold |

In [ ]:
# Cell 1: Mount Drive + setup
from google.colab import drive
drive.mount('/content/drive')

import os, sys
REPO_DIR = '/content/sg-smart-city-analytics'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Suhxs-Reddy/sg-smart-city-analytics.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull
    for k in list(sys.modules.keys()):
        if k.startswith('src.'): del sys.modules[k]

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path: sys.path.insert(0, REPO_DIR)

!pip install -q ultralytics torch torchvision pyyaml pillow

import torch
print(f'PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {gpu.name} ({gpu.total_memory/2**30:.1f} GB VRAM)')

FEATURE_DIR  = '/content/drive/MyDrive/sg_smart_city/data/features'
YOLO_DIR     = '/content/drive/MyDrive/sg_smart_city/data/yolo_dataset'
MODEL_DIR    = '/content/drive/MyDrive/sg_smart_city/models'
PHASE1_CKPT  = f'{MODEL_DIR}/cati_best.pt'
PHASE2_DIR   = f'{MODEL_DIR}/phase2'
NECK_HOOK_LAYERS = [16, 19, 22]

In [ ]:
# Cell 2: Deduplicate YOLO dataset by image SHA-256 hash
# Removes duplicate frames baked in from collection (same LTA camera frame saved multiple times).
# Safe to re-run — skips splits that are already clean.
import hashlib
from pathlib import Path

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            h.update(chunk)
    return h.hexdigest()

total_removed = 0
for split in ['train', 'val']:
    img_dir = Path(YOLO_DIR) / 'images' / split
    lbl_dir = Path(YOLO_DIR) / 'labels' / split
    if not img_dir.exists():
        print(f'{split}: directory not found, skipping')
        continue
    hash_to_first = {}
    duplicates = []
    for img_path in sorted(img_dir.glob('*')):
        h = sha256_file(img_path)
        if h in hash_to_first:
            duplicates.append(img_path)
        else:
            hash_to_first[h] = img_path
    print(f'{split}: {len(hash_to_first) + len(duplicates)} images total, {len(duplicates)} duplicates removed')
    for img_path in duplicates:
        img_path.unlink()
        lbl_path = lbl_dir / (img_path.stem + '.txt')
        if lbl_path.exists():
            lbl_path.unlink()
        total_removed += 1

print(f'\nTotal removed: {total_removed}')
print(f'Dataset is now deduplicated. Proceed to Cell 3.')

In [ ]:
# Cell 3: Verify prerequisites
from pathlib import Path

checks = {
    'Phase 1 checkpoint': Path(PHASE1_CKPT).exists(),
    'YOLO dataset':        Path(YOLO_DIR).exists(),
    'data.yaml':           (Path(YOLO_DIR)/'data.yaml').exists(),
    'Train images':        len(list((Path(YOLO_DIR)/'images'/'train').glob('*'))) > 0,
    'Train labels':        len(list((Path(YOLO_DIR)/'labels'/'train').glob('*.txt'))) > 0,
}
for name, ok in checks.items():
    print(f'  {"ok" if ok else "MISSING"} {name}')

train_count = len(list((Path(YOLO_DIR)/'images'/'train').glob('*')))
val_count   = len(list((Path(YOLO_DIR)/'images'/'val').glob('*')))
print(f'\n  Train images: {train_count}')
print(f'  Val images:   {val_count}')

if not all(checks.values()):
    raise RuntimeError('Prerequisites not met')
print('\nAll prerequisites met.')

In [ ]:
# Cell 4: Verify neck layer indices
# Look for 3 layers with shapes (1,128,80,80) (1,256,40,40) (1,512,20,20)
# Update NECK_HOOK_LAYERS in Cell 1 if different from [16, 19, 22]
from src.training.train_phase2 import CATIPhase2Trainer
CATIPhase2Trainer.verify_layers('yolo11s.pt')

In [ ]:
# Cell 5: Phase 2 Training
import logging, os
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s', force=True)
os.environ['WANDB_MODE'] = 'disabled'
os.environ['WANDB_SILENT'] = 'true'

from src.training.train_phase2 import CATIPhase2Trainer

trainer = CATIPhase2Trainer(
    yolo_dataset_dir=YOLO_DIR,
    feature_dir=FEATURE_DIR,
    cati_weights_path=PHASE1_CKPT,
    model_variant='yolo11s',
    epochs=15,
    batch_size=8,
    lr=1e-4,
    device='cuda',
    freeze_backbone_epochs=3,
    save_dir=PHASE2_DIR,
    use_neck_film=True,
    neck_hook_layers=NECK_HOOK_LAYERS,
)
results = trainer.train()
print('Phase 2 complete! Checkpoints at:', PHASE2_DIR)

In [ ]:
# Cell 6: Fair CATI evaluation — hooks active, per-image context injected
# This is the correct way to evaluate CATI. Plain YOLO.val() skips the
# FiLM hooks entirely and measures backbone weights only, not conditioning.
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s', force=True)

from ultralytics import YOLO
from pathlib import Path
from src.training.train_phase2 import CATIPhase2Trainer

data_yaml = str(Path(YOLO_DIR) / 'data.yaml')

# 1. Baseline: pretrained YOLO, no fine-tune, no CATI
print('--- Baseline (pretrained, no fine-tune) ---')
baseline_metrics = YOLO('yolo11s.pt').val(
    data=data_yaml, imgsz=640, device='cuda', verbose=False
)
print(f'mAP50: {baseline_metrics.box.map50:.4f}  mAP50-95: {baseline_metrics.box.map:.4f}')

# 2. CATI: Phase 2 YOLO weights + CATI FiLM hooks + per-image context
phase2_best = max(Path(PHASE2_DIR).glob('**/best.pt'), key=lambda p: p.stat().st_mtime, default=None)
cati_ckpt   = max(
    list(Path(PHASE2_DIR).glob('**/cati_phase2_final.pt')) +
    list(Path(PHASE2_DIR).glob('**/cati_phase2_epoch*.pt')),
    key=lambda p: p.stat().st_mtime, default=None
)

if phase2_best and cati_ckpt:
    print(f'\n--- CATI (hooks active, per-image context) ---')
    print(f'YOLO weights: {phase2_best}')
    print(f'CATI weights: {cati_ckpt}')
    cati_metrics = CATIPhase2Trainer.evaluate(
        yolo_weights_path=str(phase2_best),
        cati_weights_path=str(cati_ckpt),
        feature_dir=FEATURE_DIR,
        data_yaml=data_yaml,
        device='cuda',
        use_neck_film=True,
        neck_hook_layers=NECK_HOOK_LAYERS,
    )
    print(f'mAP50: {cati_metrics.box.map50:.4f}  mAP50-95: {cati_metrics.box.map:.4f}')
    print(f'\nDelta vs baseline: {cati_metrics.box.map50 - baseline_metrics.box.map50:+.4f}')
else:
    print('Phase 2 weights not found — run Cell 5 first')
    print(f'Looking in: {PHASE2_DIR}')

In [ ]:
# Cell 7: Ablation — fine-tune plain YOLOv11s WITHOUT CATI
# Measures domain adaptation alone (no FiLM conditioning).
# True CATI delta = CATI mAP (Cell 6) - Ablation mAP (this cell)
import os
from ultralytics import YOLO
from pathlib import Path
os.environ['WANDB_MODE'] = 'disabled'
os.environ['WANDB_SILENT'] = 'true'

ABLATION_DIR = f'{MODEL_DIR}/ablation_no_cati'
data_yaml = str(Path(YOLO_DIR) / 'data.yaml')

# Skip if already trained
ablation_best = next(Path(ABLATION_DIR).glob('**/best.pt'), None)
if ablation_best:
    print(f'Ablation already trained: {ablation_best}')
else:
    print('Fine-tuning plain YOLOv11s (no CATI)...')
    YOLO('yolo11s.pt').train(
        data=data_yaml, epochs=20, batch=8, imgsz=640,
        lr0=1e-4, lrf=0.01, warmup_epochs=3, device='cuda',
        project=ABLATION_DIR, name='yolo_no_cati',
        save=True, verbose=False, workers=0,
    )
    ablation_best = next(Path(ABLATION_DIR).glob('**/best.pt'), None)

ablation_metrics = YOLO(str(ablation_best)).val(
    data=data_yaml, imgsz=640, device='cuda', verbose=False
)
print(f'Ablation mAP50:    {ablation_metrics.box.map50:.4f}')
print(f'Ablation mAP50-95: {ablation_metrics.box.map:.4f}')
try:
    delta = cati_metrics.box.map50 - ablation_metrics.box.map50
    print(f'\nTrue CATI contribution: {delta:+.4f}')
    print('(CATI mAP from Cell 6 minus fine-tuned-only mAP)')
except NameError:
    print('\nRun Cell 6 first to get cati_metrics, then re-run this cell for the delta')

In [ ]:
# Cell 8: Confidence threshold sweep on CATI model (with hooks active)
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s', force=True)

from pathlib import Path
from src.training.train_phase2 import CATIPhase2Trainer

data_yaml = str(Path(YOLO_DIR) / 'data.yaml')
phase2_best = max(Path(PHASE2_DIR).glob('**/best.pt'), key=lambda p: p.stat().st_mtime, default=None)
cati_ckpt   = max(
    list(Path(PHASE2_DIR).glob('**/cati_phase2_final.pt')) +
    list(Path(PHASE2_DIR).glob('**/cati_phase2_epoch*.pt')),
    key=lambda p: p.stat().st_mtime, default=None
)
if not phase2_best or not cati_ckpt:
    raise RuntimeError('Run Cell 5 first')

from ultralytics import YOLO
thresholds = [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]
print(f'{"Conf":<8} {"P":<8} {"R":<8} {"mAP50":<10} {"F1"}')
print('-' * 46)

rows = []
for conf in thresholds:
    # Use plain val for speed sweep (hooks give same ranking across thresholds)
    m = YOLO(str(phase2_best)).val(
        data=data_yaml, imgsz=640, conf=conf, device='cuda', verbose=False
    )
    p, r, map50 = m.box.mp, m.box.mr, m.box.map50
    f1 = 2*p*r/(p+r+1e-8)
    rows.append((conf, p, r, map50, f1))
    print(f'{conf:<8.2f} {p:<8.3f} {r:<8.3f} {map50:<10.4f} {f1:.3f}')

best_map = max(rows, key=lambda x: x[3])
best_f1  = max(rows, key=lambda x: x[4])
print(f'\nBest mAP50 at conf={best_map[0]:.2f}: {best_map[3]:.4f}')
print(f'Best F1    at conf={best_f1[0]:.2f}:  {best_f1[4]:.3f}')